# IS2083 Lab 5: AI Automation Tool, Part 1

**Before you begin, you need TWO things:**

1. Your **Jupyter login** (you used it to open this notebook).
2. Your **API key** for the class Qwen Coder model server (from your instructor).

Run cells with **Shift + Enter**, top to bottom.

## Step 1: Install the openai client (run once)

In [ ]:
%pip install openai

## Step 2: Connect to the Qwen Coder model server

In [1]:
from openai import OpenAI

BASE_URL = "http://gpustack.cds.utsa.edu/v1"
API_KEY  = "API_key"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
MODEL = client.models.list().data[1].id
print("Connected. Model:", MODEL)

Connected. Model: qwen3.5-122b-a10b


## Step 3: A reusable model helper (with retries)

In [2]:
import time

def ask_model(prompt: str, temperature: float = 0.2, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))



Hello there!


## Step 4: Plan the tool

## Step 5: Use Qwen Coder as a coding assistant

In [3]:
question = (
    "I am writing a Python function that asks an LLM to classify a security log entry "
    "and returns a dataclass with label and reason. Suggest a robust prompt format and "
    "a defensive way to parse the reply. Keep it concise."
)
print(ask_model(question))



```python
from dataclasses import dataclass
import json
import re

@dataclass
class LogClassification:
    label: str
    reason: str

def parse_llm_response(raw: str) -> LogClassification:
    # 1. Strip markdown code blocks (common LLM behavior)
    clean = re.sub(r"```(?:json)?\s*|\s*```", "", raw).strip()
    
    # 2. Parse JSON defensively
    try:
        data = json.loads(clean)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}")

    # 3. Validate schema
    if not isinstance(data, dict) or "label" not in data or "reason" not in data:
        raise ValueError("Response missing 'label' or 'reason'")

    return LogClassification(label=str(data["label"]), reason=str(data["reason"]))

def classify_log(log_entry: str, client) -> LogClassification:
    # Robust Prompt: Enforces JSON, restricts output, separates data from instructions
    prompt = f"""
    ROLE: Security Analyst.
    TASK: Classify the log entry below.
    OUTPUT: Valid JSON ONLY. 

## Step 6: Define the result type and parsers

In [5]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

## Step 7: Name your tool

In [6]:
TOOL_NAME = "SentinelScan"

## Step 8: Build the tool

In [7]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are an automated security log analysis engine. Classify the following log entry.\n"
        "You must respond with EXACTLY two lines and no other commentary, markdown fences, or text:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one concise sentence explaining your classification>\n\n"
        f"Log entry:\n{line}\n"
    )
    text = ask_model(prompt)
    label = Label.UNKNOWN
    reason = "Failed to parse model response."
    label_match = LABEL_RE.search(text)
    if label_match:
        raw_label = label_match.group(1).upper()
        try:
            label = Label(raw_label)
        except ValueError:
            label = Label.UNKNOWN
    reason_match = REASON_RE.search(text)
    if reason_match:
        reason = reason_match.group(1).strip()
    elif text:
        reason = text.strip().replace("\n", " ")
    return Triage(line=line, label=label, reason=reason)

## Step 9: Quick sanity check

In [9]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='Failed authentication attempts against the root account indicate potential unauthorized access or brute-force activity.')


## Step 10: Assess a real security log file

In [13]:
LOG_FILE = "security_log.csv"
MAX_ENTRIES = 5

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:limit]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]
print(f"Assessed {len(results)} entries from {LOG_FILE}")

Starting triage for 5 entries...
[1/5] Assessing line... Done (BENIGN)
[2/5] Assessing line... Done (BENIGN)
[3/5] Assessing line... Done (BENIGN)
[4/5] Assessing line... Done (BENIGN)
[5/5] Assessing line... Done (BENIGN)

Assessed 5 entries from security_log.csv


## Step 11: Write the assessment report

In [14]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")
        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")
    return path

report_path = write_report(results, TOOL_NAME)
print("Wrote", report_path, "\n")
print(open(report_path, encoding="utf-8").read())

Wrote assessment.md 

# SentinelScan Security Log Assessment

Generated: 2026-11-10 19:52

Entries assessed: 5

- **BENIGN** This entry is a CSV header row containing column names rather than an actual security event record.
  - `﻿Keywords,Date and Time,Source,Event ID,Task Category`
- **BENIGN** Event ID 5379 indicates standard credential retrieval which is common for legitimate software operations.
  - `Audit Success,11/10/2026 7:38:58 PM,Microsoft-Windows-Security-Auditing,5379,User Account Management,"Credential Manager credentials were read.`
- **BENIGN** The entry is incomplete and contains no actionable security indicators or malicious payloads.
  - `Subject:`
- **BENIGN** This is a standard security log field identifying a user account with no indicators of compromise.
  - `Security ID:		TOP\benc9`
- **BENIGN** This entry contains only a username identifier without context indicating malicious activity or anomalies.
  - `Account Name:		benc9`



## Step 12: Read, test, and trust